# Iceberg Basics — 01: Your First Iceberg Table

## What you will learn
Apache Iceberg is an **open table format** — not a file format, not a storage engine.
It adds ACID, time travel, and schema evolution on top of Parquet/ORC/Avro files.

In this notebook:
1. What Iceberg is and how it differs from Delta Lake
2. The Iceberg metadata architecture — metadata.json, manifests, data files
3. Creating an Iceberg table — SQL and DataFrame API
4. Reading an Iceberg table
5. The Hadoop catalog — how tables are discovered in our setup
6. Iceberg vs Delta vs Hive — key differences


In [1]:
import os, time, pathlib, shutil, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/iceberg_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('iceberg-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark.sql("CREATE DATABASE IF NOT EXISTS local.icedb")
print(f"Spark {spark.version} | Iceberg catalog ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:24:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Iceberg catalog ready


## Step 1 — Iceberg Architecture

```
Iceberg Table (logical)
        │
        ▼
Catalog (local Hadoop catalog)
  └── metadata pointer → current metadata.json

metadata/
  v1.metadata.json   ← table schema, partition spec, snapshot list
  v2.metadata.json   ← after first write (new current)
  snap-001-1.avro    ← manifest list (snapshot 1)
  snap-001-m1.avro   ← manifest file (list of data files)

data/
  category=Electronics/
    part-00001.parquet   ← actual data
  category=Books/
    part-00002.parquet

Key difference from Delta:
  Delta   → _delta_log/*.json (JSON commit log)
  Iceberg → metadata/*.json + manifest *.avro files
```


In [2]:
import os, time, pathlib, shutil, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/iceberg_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('iceberg-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark.sql("CREATE DATABASE IF NOT EXISTS local.icedb")
print(f"Spark {spark.version} | Iceberg catalog ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

Spark 4.0.2 | Iceberg catalog ready


26/04/10 20:24:34 WARN TaskSetManager: Stage 0 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:24:36 WARN TaskSetManager: Stage 1 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.


Dataset ready: 100,000 rows


## Step 2 — Create an Iceberg Table


In [3]:
# Drop if exists
spark.sql("DROP TABLE IF EXISTS local.icedb.orders")

# SQL CREATE TABLE
spark.sql("""
    CREATE TABLE local.icedb.orders (
        order_id    BIGINT,
        customer_id BIGINT,
        product     STRING,
        category    STRING,
        country     STRING,
        quantity    INT,
        price       DOUBLE,
        revenue     DOUBLE,
        status      STRING,
        order_date  DATE
    )
    USING iceberg
    PARTITIONED BY (months(order_date))
    TBLPROPERTIES (
        'write.format.default'            = 'parquet',
        'write.parquet.compression-codec' = 'zstd'
    )
""")
print("Table created: local.icedb.orders")
spark.sql("DESCRIBE EXTENDED local.icedb.orders").show(30, truncate=False)

Table created: local.icedb.orders
+----------------------------+------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                                           |comment|
+----------------------------+------------------------------------------------------------------------------------------------------------------------------------+-------+
|order_id                    |bigint                                                                                                                              |NULL   |
|customer_id                 |bigint                                                                                                                              |NULL   |
|product                     |string                                                                      

In [4]:
# Insert data
df.writeTo("local.icedb.orders").append()
count = spark.table("local.icedb.orders").count()
print(f"Inserted: {count:,} rows")

# Read back
spark.table("local.icedb.orders").show(5)
spark.table("local.icedb.orders").printSchema()

26/04/10 20:24:38 WARN TaskSetManager: Stage 4 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Inserted: 100,000 rows
+--------+-----------+-----------+--------+-------+--------+------+-------+---------+----------+
|order_id|customer_id|    product|category|country|quantity| price|revenue|   status|order_date|
+--------+-----------+-----------+--------+-------+--------+------+-------+---------+----------+
|       5|       1675| Product_24|    Food|     UK|       3|219.24| 657.72|  shipped|2024-06-21|
|       9|       3433|Product_172|   Books|     UK|       6|166.84|1001.04|cancelled|2024-06-30|
|      39|       3911| Product_68|    Food|     DE|       4|671.38|2685.52|  shipped|2024-06-09|
|      62|       7619|Product_114|    Food|     FR|      10|608.63| 6086.3|cancelled|2024-06-13|
|      75|       6390|Product_108|  Sports|     FR|       1|943.03| 943.03|delivered|2024-06-01|
+--------+-----------+-----------+--------+-------+--------+------+-------+---------+----------+
only showing top 5 rows
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = tr

## Step 3 — Inspect Iceberg Metadata


In [5]:
# Snapshots — one per write operation
print("Snapshots:")
spark.sql("SELECT snapshot_id, committed_at, operation, summary FROM local.icedb.orders.snapshots").show(truncate=False)

# Files — the physical Parquet files
print("Data files:")
spark.sql("""
    SELECT file_path, file_size_in_bytes/1024 AS size_kb, record_count, partition
    FROM local.icedb.orders.files
    LIMIT 5
""").show(truncate=False)

# Partitions
print("Partitions:")
spark.sql("SELECT partition, record_count, file_count FROM local.icedb.orders.partitions ORDER BY record_count DESC LIMIT 6").show()

Snapshots:
+-------------------+-----------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |committed_at           |operation|summary                                                                                                                                                                                                                                                                                                                                                                                                 

## Step 4 — Where Iceberg Stores Its Files


In [6]:
import pathlib, os

warehouse = "/workspace/data/warehouse/iceberg"
print("Iceberg warehouse structure:")
for root, dirs, files in os.walk(warehouse):
    level = root.replace(warehouse, '').count(os.sep)
    if level > 4: continue
    indent = '  ' * level
    folder = os.path.basename(root)
    n_files = len(files)
    if level > 0:
        print(f"{indent}{folder}/  [{n_files} file(s)]")

print()
print("Key directories:")
print("  metadata/ → JSON metadata files + Avro manifest files")
print("  data/     → actual Parquet data files")
print()
print("Unlike Delta (_delta_log/), Iceberg uses Avro for manifests")
print("which makes them more compact for large tables with many files.")

Iceberg warehouse structure:
  demo/  [0 file(s)]
    employees_ice/  [0 file(s)]
      data/  [24 file(s)]
      metadata/  [54 file(s)]
  events_db/  [0 file(s)]
    clickstream/  [0 file(s)]
      data/  [0 file(s)]
        event_ts_hour=2026-04-08-15/  [8 file(s)]
      metadata/  [28 file(s)]
    event_summary/  [0 file(s)]
      data/  [104 file(s)]
      metadata/  [26 file(s)]
  icedb/  [0 file(s)]
    orders/  [0 file(s)]
      data/  [0 file(s)]
        order_date_month=2024-01/  [2 file(s)]
        order_date_month=2024-02/  [2 file(s)]
        order_date_month=2024-03/  [2 file(s)]
        order_date_month=2024-04/  [2 file(s)]
        order_date_month=2024-05/  [2 file(s)]
        order_date_month=2024-06/  [2 file(s)]
        order_date_month=2024-07/  [2 file(s)]
        order_date_month=2024-08/  [2 file(s)]
        order_date_month=2024-09/  [2 file(s)]
        order_date_month=2024-10/  [2 file(s)]
        order_date_month=2024-11/  [2 file(s)]
        order_date_mont

## Step 5 — Iceberg vs Delta: Side by Side


In [7]:
# Create equivalent Delta table for comparison
from delta.tables import DeltaTable

DELTA_PATH = f"{DATA_DIR}/delta_compare"
shutil.rmtree(DELTA_PATH, ignore_errors=True)
df.write.format("delta").mode("overwrite").save(DELTA_PATH)

print("Feature comparison:")
print()
features = [
    ("ACID transactions",          "✅ Yes", "✅ Yes"),
    ("Time travel",                "✅ Yes", "✅ Yes"),
    ("Schema evolution",           "✅ Rich", "✅ Good"),
    ("Partition evolution",        "✅ Yes (no rewrite)", "⚠️  Limited"),
    ("Hidden partitioning",        "✅ Yes", "❌ No"),
    ("Multi-engine support",       "✅ Spark/Trino/Flink", "⚠️  Spark-first"),
    ("Metadata format",            "JSON + Avro manifests", "JSON commits"),
    ("Branching / tagging",        "✅ Native", "⚠️  Not built-in"),
    ("OPTIMIZE/ZORDER",            "❌ Not built-in", "✅ Native"),
    ("Change Data Feed",           "Via snapshot diff", "✅ Native CDF"),
    ("Liquid Clustering",          "❌ Not available", "✅ Delta 3.1+"),
]

print(f"  {'Feature':<35} {'Iceberg':>25} {'Delta':>25}")
print("  " + "-" * 87)
for feat, ice, dlt in features:
    print(f"  {feat:<35} {ice:>25} {dlt:>25}")

26/04/10 20:24:47 WARN TaskSetManager: Stage 14 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Feature comparison:

  Feature                                               Iceberg                     Delta
  ---------------------------------------------------------------------------------------
  ACID transactions                                       ✅ Yes                     ✅ Yes
  Time travel                                             ✅ Yes                     ✅ Yes
  Schema evolution                                       ✅ Rich                    ✅ Good
  Partition evolution                        ✅ Yes (no rewrite)               ⚠️  Limited
  Hidden partitioning                                     ✅ Yes                      ❌ No
  Multi-engine support                      ✅ Spark/Trino/Flink           ⚠️  Spark-first
  Metadata format                         JSON + Avro manifests              JSON commits
  Branching / tagging                                  ✅ Native          ⚠️  Not built-in
  OPTIMIZE/ZORDER                                ❌ Not built-in                

## Summary

### What Iceberg brings
- **Open spec** — not tied to any vendor, works with any engine
- **Partition evolution** — change partition scheme without rewriting data
- **Hidden partitioning** — no partition columns in SQL queries
- **Branching/tagging** — Git-like workflow for data

### Catalog configuration (already in spark-defaults.conf)
```
spark.sql.extensions = org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
spark.sql.catalog.local = org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.local.type = hadoop
spark.sql.catalog.local.warehouse = /workspace/data/warehouse/iceberg
```

**Next:** `02_reading_writing.ipynb` — all read/write options.
